---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Persiapan: SparkSession & Dataset

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, abs

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [6]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Membaca dataset

In [7]:
df = spark.read.csv("tugas11_penjualan.csv",
                    header=True,
                    inferSchema=True)

print("Jumlah data:", df.count())
df.show(5)

Jumlah data: 2500
+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



### A. Feature Engineering

In [8]:
# StringIndexer
from pyspark.ml.feature import StringIndexer

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

model_menu = indexer_menu.fit(df)
df_indexed = model_menu.transform(df)

model_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_cabang.transform(df_indexed)

model_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_waktu.transform(df_indexed)

df_indexed.select(
    "nama_menu", "nama_menu_idx",
    "cabang", "cabang_idx",
    "waktu_pesan", "waktu_pesan_idx"
).show(8)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 8 rows



In [9]:
#VectorAssembler
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx",
               "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)

df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



### B. Train/Test Split dan Pelatihan Model

In [10]:
# Membagi dataset
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

Jumlah data training : 1839
Jumlah data testing  : 661


In [11]:
# Membuat dan melatih model
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan"
)

model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


### C. Evaluasi Model

In [12]:
# Melakukan prediksi
predictions = model.transform(test)

predictions.select(
    "qty",
    "harga_satuan",
    "total_penjualan",
    "prediction"
).show(10)

+---+------------+------------------+------------------+
|qty|harga_satuan|   total_penjualan|        prediction|
+---+------------+------------------+------------------+
|  1|       25000|19721.170446051627|31056.146105506836|
|  1|       25000|  21236.0956379149|31056.146105506836|
|  4|       22000|  85834.8564401191| 88127.37331410211|
|  4|       22000|102174.09100317833| 88127.37331410211|
|  4|       30000|113044.63431555482|112310.74025938749|
|  5|       22000|115810.29600214415|110174.03658512786|
|  5|       25000|140386.40273615732|119242.79918960987|
|  2|       30000| 73842.64420790318| 67877.77492201277|
|  3|       15000|32345.642983082646| 44580.62517062847|
|  4|       15000| 45510.16712269087| 66627.28844165422|
+---+------------+------------------+------------------+
only showing top 10 rows



In [13]:
# Menghitung RMSE dan R²
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f} ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,901
R²   : 0.9107 (91.1% variasi data berhasil dijelaskan model)


### D. Eksperimen Perbandingan Fitur

In [14]:
# Membuat VectorAssembler kedua
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

df_sederhana = assembler_sederhana.transform(df_indexed)

df_sederhana.select(
    "features_sederhana",
    "total_penjualan"
).show(5, truncate=False)

+------------------+------------------+
|features_sederhana|total_penjualan   |
+------------------+------------------+
|[1.0,30000.0]     |27750.163531558544|
|[3.0,20000.0]     |64216.16074371864 |
|[5.0,22000.0]     |108722.79259154423|
|[2.0,20000.0]     |54483.205871792496|
|[2.0,30000.0]     |58157.94412015849 |
+------------------+------------------+
only showing top 5 rows



In [15]:
# Membagi dataset dan melatih model kedua
train_sederhana, test_sederhana = df_sederhana.randomSplit(
    [0.75, 0.25],
    seed=101
)

lr_sederhana = LinearRegression(
    featuresCol="features_sederhana",
    labelCol="total_penjualan"
)

model_sederhana = lr_sederhana.fit(train_sederhana)

print("Model kedua berhasil dilatih!")

Model kedua berhasil dilatih!


In [16]:
# Evaluasi dan Perbandingan R²
predictions_sederhana = model_sederhana.transform(test_sederhana)

evaluator_r2_sederhana = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2_sederhana = evaluator_r2_sederhana.evaluate(predictions_sederhana)

print("R² Model Utama    :", round(r2, 4))
print("R² Model Sederhana:", round(r2_sederhana, 4))

R² Model Utama    : 0.9107
R² Model Sederhana: 0.9108


### E. Prediksi Skenario Baru

In [17]:
# Membuat data skenario baru
data_baru = [
    ("Kopi Susu", "Semarang", "Pagi", 2, 18000),
    ("Americano", "Magelang", "Siang", 3, 20000),
    ("Roti Bakar", "Yogyakarta", "Malam", 1, 25000)
]

kolom = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]

df_baru = spark.createDataFrame(data_baru, kolom)

df_baru.show()

+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Semarang|       Pagi|  2|       18000|
| Americano|  Magelang|      Siang|  3|       20000|
|Roti Bakar|Yogyakarta|      Malam|  1|       25000|
+----------+----------+-----------+---+------------+



In [18]:
# Transformasi menggunakan StringIndexer yang sama
df_baru_indexed = model_menu.transform(df_baru)
df_baru_indexed = model_cabang.transform(df_baru_indexed)
df_baru_indexed = model_waktu.transform(df_baru_indexed)

df_baru_indexed.select(
    "nama_menu", "nama_menu_idx",
    "cabang", "cabang_idx",
    "waktu_pesan", "waktu_pesan_idx"
).show()

+----------+-------------+----------+----------+-----------+---------------+
| nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+----------+-------------+----------+----------+-----------+---------------+
| Kopi Susu|          1.0|  Semarang|       2.0|       Pagi|            0.0|
| Americano|          0.0|  Magelang|       0.0|      Siang|            2.0|
|Roti Bakar|          4.0|Yogyakarta|       1.0|      Malam|            1.0|
+----------+-------------+----------+----------+-----------+---------------+



In [19]:
# Menggabungkan fitur dan melakukan prediksi
df_baru_final = assembler.transform(df_baru_indexed)

hasil_prediksi = model.transform(df_baru_final)

hasil_prediksi.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Semarang  |Pagi       |2  |18000       |32038.0792973317  |
|Americano |Magelang  |Siang      |3  |20000       |60374.50710207816 |
|Roti Bakar|Yogyakarta|Malam      |1  |25000       |30842.573606177306|
+----------+----------+-----------+---+------------+------------------+

